# 노트북 1 — 데이터 준비 (PhysioNet EEG Motor Imagery)

**EEG Resonate Encoding SNN 프로젝트** · 노트북 1

PhysioNet EEG Motor Movement/Imagery 데이터셋에서 왼손·오른손 주먹 씁기 심상 구간을 내려받아
전처리하고, 단일 창 길이가 아니라 **두 가지 창 길이(0.5–2.0s, 0.5–3.5s)**로 각각 에폭을 잘라
`.npz` 파일로 저장한다. 피험자는 40명(S1~S40)으로 확대해서, 노트북 2의 CSP+LDA 선별 관문을
통과하는 피험자만 이후 단계에서 쓴다.

### 학습 목표
- `mne.datasets.eegbci`로 피험자 40명의 EDF 데이터를 내려받는다.
- 9채널만 골라 공통평균재참조(CAR) → 1–45 Hz 대역통과 → 128 Hz 리샘플 → 채널별 z-score를 적용한다.
- 전처리된 연속 신호에서 큐 후 **0.5–2.0s**와 **0.5–3.5s** 두 가지 구간을 각각 에폭으로 잘라낸다.
- 두 창 길이 결과를 각각 `.npz`로 저장하고, 피험자당 시행 수와 클래스 균형을 확인한다.


## 1. 설치와 임포트

In [1]:
%pip install -q mne

import os
import numpy as np
import mne
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from mne import Epochs, events_from_annotations
import matplotlib.pyplot as plt
import koreanize_matplotlib          # 그래프 한글 폰트

np.random.seed(0)                    # 재현성
mne.set_log_level('WARNING')         # MNE 로그 출력 억제

print('mne 버전:', mne.__version__)


Note: you may need to restart the kernel to use updated packages.


mne 버전: 1.12.1


## 2. 파라미터 설정

실험 설계(CLAUDE.md 3, 4절)에 따른 상수들이다. 창 길이는 짧은 버전(SHORT, 원래 스펙)과 긴 버전(LONG)을 둘 다 정의한다.

In [2]:
SUBJECTS = range(1, 41)                                                  # 피험자 40명 (S1~S40)
RUNS = [4, 8, 12]                                                        # 왼손/오른손 주먹 쥐기 심상 러닝
CHANNELS = ['FC3', 'FCz', 'FC4', 'C3', 'Cz', 'C4', 'CP3', 'CPz', 'CP4']  # 9채널
L_FREQ, H_FREQ = 1.0, 45.0                                               # 대역통과 범위 [Hz]
FS = 128                                                                 # 리샘플 주파수 [Hz]
DATA_DIR = '../data/raw'                                                 # EDF 원본 저장 경로

WINDOWS = {
    'short': {'tmin': 0.5, 'tmax': 2.0, 'num_steps': 192, 'out_path': '../data/processed/eeg_epochs.npz'},
    'long':  {'tmin': 0.5, 'tmax': 3.5, 'num_steps': 384, 'out_path': '../data/processed/eeg_epochs_long.npz'},
}                                                                         # 짧은 창(원래 스펙) / 긴 창


## 3. 피험자별 전처리 함수

원본 불러오기·채널명 표준화·CAR·대역통과·리샘플·z-score는 창 길이와 무관하게 한 번만 수행하고(`load_and_preprocess_raw`),
그 결과를 길이가 다른 두 구간으로 각각 에폭화한다(`epoch_subject`).

In [3]:
def load_and_preprocess_raw(subject_id):
    raw_fnames = eegbci.load_data(subject_id, RUNS, path=DATA_DIR, update_path=True)  # EDF 내려받기
    raws = [read_raw_edf(f, preload=True) for f in raw_fnames]
    raw = concatenate_raws(raws)                      # 3개 런을 하나로 이어붙임

    eegbci.standardize(raw)                           # 채널명 표준화 (Fc3. → FC3)
    raw.set_montage('standard_1005')                  # 좌표계 설정
    raw.set_eeg_reference('average')                  # 공통 평균 기준 재참조 (전체 채널로)
    raw.pick(CHANNELS)                                 # 9채널만 선택

    raw.filter(L_FREQ, H_FREQ, fir_design='firwin')    # 1-45 Hz 대역통과
    raw.resample(FS)                                   # 128 Hz 리샘플
    raw.apply_function(
        lambda x: (x - x.mean()) / x.std(),
        channel_wise=True
    )                                                   # 채널별 z-score
    return raw


def epoch_subject(raw, tmin, tmax, num_steps):
    events, event_id_all = events_from_annotations(raw)
    event_id = {name: event_id_all[name] for name in ('T1', 'T2')}   # T1=왼손, T2=오른손 (T0=휴식 제외)

    epochs = Epochs(
        raw, events, event_id=event_id,
        tmin=tmin, tmax=tmax, baseline=None, preload=True
    )

    data = epochs.get_data()[:, :, :num_steps].astype(np.float32)      # (시행, 채널, 시간)
    labels = (epochs.events[:, 2] == event_id['T2']).astype(np.int64)  # 오른손=1, 왼손=0
    return data, labels


## 4. 전체 피험자 처리 — 윈도우 2개를 함께 수집

In [4]:
collected = {window_name: {'X': [], 'y': [], 'subject_ids': []} for window_name in WINDOWS}

for subject_id in SUBJECTS:
    raw = load_and_preprocess_raw(subject_id)

    for window_name, cfg in WINDOWS.items():
        data, labels = epoch_subject(raw, cfg['tmin'], cfg['tmax'], cfg['num_steps'])
        collected[window_name]['X'].append(data)
        collected[window_name]['y'].append(labels)
        collected[window_name]['subject_ids'].append(np.full(labels.shape, subject_id, dtype=np.int64))

    print(f'S{subject_id:02d} 완료')


S01 완료


S02 완료


S03 완료


S04 완료


S05 완료


...

S36 완료


S37 완료


S38 완료


S39 완료


S40 완료


## 5. 저장 및 확인 — 윈도우별 시행 수·클래스 균형

In [5]:
os.makedirs('../data/processed', exist_ok=True)

for window_name, cfg in WINDOWS.items():
    X = np.concatenate(collected[window_name]['X'], axis=0)
    y = np.concatenate(collected[window_name]['y'], axis=0)
    subject_ids = np.concatenate(collected[window_name]['subject_ids'], axis=0)

    np.savez(
        cfg['out_path'],
        X=X, y=y, subject_ids=subject_ids,
        channels=np.array(CHANNELS), fs=FS, tmin=cfg['tmin'], tmax=cfg['tmax']
    )

    n_subjects = len(set(subject_ids.tolist()))
    print(f"[{window_name} {cfg['tmin']}-{cfg['tmax']}s] 저장: {cfg['out_path']}")
    print(f'  전체 형태: {X.shape} | 피험자 수: {n_subjects} | 평균 시행/피험자: {len(y)/n_subjects:.1f} '
          f'| 오른손 비율: {y.mean():.2f}')


[short 0.5-2.0s] 저장: ../data/processed/eeg_epochs.npz
  전체 형태: (1800, 9, 192) | 피험자 수: 40 | 평균 시행/피험자: 45.0 | 오른손 비율: 0.50
[long 0.5-3.5s] 저장: ../data/processed/eeg_epochs_long.npz
  전체 형태: (1800, 9, 384) | 피험자 수: 40 | 평균 시행/피험자: 45.0 | 오른손 비율: 0.50


## 정리

- `mne.datasets.eegbci`로 피험자 40명(S1~S40)의 왼손/오른손 주먹 씁기 심상 데이터를 내려받았다.
- 9채널 → 공통평균재참조(CAR) → 1–45 Hz 대역통과 → 128 Hz 리샘플 → 채널별 z-score 순으로 전처리했다.
- 큐 후 **0.5–2.0s**(짧은 창, T=192)와 **0.5–3.5s**(긴 창, T=384) 두 가지로 에폭화해 각각
  `data/processed/eeg_epochs.npz`, `data/processed/eeg_epochs_long.npz`로 저장했다.

다음 노트북(02_sanity_check)에서는 이 두 창 길이 데이터에 CSP+LDA 선별 관문을 적용해 피험자를 추린다.